In [ ]:
import pandas as pd
from itertools import combinations

# Load the dataset
data_path = 'household_consumption.csv'
df = pd.read_csv(data_path)

# Preprocess the Active Devices column to create a list of transactions
df['Active Devices'] = df['Active Devices'].apply(lambda x: x.split(', '))
transactions = df['Active Devices'].tolist()

# Get a list of all unique devices
devices = sorted({device for transaction in transactions for device in transaction})

# Helper function to calculate support
def calculate_support(itemset, transactions):
    return sum(1 for transaction in transactions if itemset.issubset(transaction)) / len(transactions)

# Generate frequent itemsets using the Apriori algorithm
def apriori(transactions, min_support=0.2):
    transactions = [set(transaction) for transaction in transactions]
    freq_itemsets = []
    k = 1
    current_itemsets = [frozenset([device]) for device in devices]

    while current_itemsets:
        # Calculate support for current itemsets
        itemset_supports = {
            itemset: calculate_support(itemset, transactions) for itemset in current_itemsets
        }
        # Filter itemsets based on min_support
        current_itemsets = [
            itemset for itemset, support in itemset_supports.items() if support >= min_support
        ]
        # Add to the frequent itemsets list
        freq_itemsets.extend([
            (itemset, itemset_supports[itemset]) for itemset in current_itemsets
        ])
        # Generate new candidate itemsets (k+1)
        current_itemsets = [
            frozenset(set1 | set2) for set1, set2 in combinations(current_itemsets, 2)
            if len(set1 | set2) == k + 1
        ]
        k += 1
    return freq_itemsets

# Allocate time and power based on sorted itemsets
def allocate_time_and_power(freq_itemsets, power_limit, total_time, min_time_threshold, power_per_device):
    # Sort itemsets by number of elements, then by support
    freq_itemsets = sorted(freq_itemsets, key=lambda x: (-len(x[0]), -x[1]))

    allocation = {}
    remaining_power = power_limit
    remaining_time = total_time

    for itemset, support in freq_itemsets:
        # Sort devices in itemset by their singular support value
        sorted_devices = sorted(itemset, key=lambda x: next(s[1] for s in freq_itemsets if s[0] == frozenset([x])), reverse=True)

        feasible = True
        for device in sorted_devices:
            if device in allocation:
                continue

            power_required = power_per_device[device] * remaining_time

            # Check if we can allocate time for this device within remaining power
            if power_required <= remaining_power:
                allocation[device] = remaining_time
                remaining_power -= power_required
            else:
                # Allocate proportional time based on remaining power
                max_time = remaining_power / power_per_device[device]
                if max_time >= min_time_threshold:
                    allocation[device] = max_time
                    remaining_power -= max_time * power_per_device[device]
                else:
                    feasible = False
                    break

        if not feasible:
            break

    # Remove devices below the minimum time threshold
    allocation = {device: time for device, time in allocation.items() if time >= min_time_threshold}

    return allocation

# Parameters
min_support = 0.2
min_confidence = 0.5
power_limit = 5  # kW
total_time = 1  # hours
min_time_threshold = 0.25  # 15 minutes
power_per_device = {
    'Refrigerator': 1,    # kW/hour
    'TV': 2,              # kW/hour
    'Microwave': 2,       # kW/hour
    'Washing Machine': 3,  # kW/hour
    'Lighting': 0.5,      # kW/hour
    'Computer': 1.5,      # kW/hour
    'Oven': 1             # kW/hour
}

# Run Apriori and allocate time and power
frequent_itemsets = apriori(transactions, min_support=min_support)
allocation = allocate_time_and_power(frequent_itemsets, power_limit, total_time, min_time_threshold, power_per_device)

# Print results
print("Device Allocation:")
for device, time in allocation.items():
    print(f"{device}: {time:.2f} hours")

# Save frequent itemsets
frequent_itemsets_df = pd.DataFrame(frequent_itemsets, columns=['Itemset', 'Support'])
# frequent_itemsets_df.to_csv('{path_to_save}/frequent_itemsets.csv', index=False)


Device Allocation:
Refrigerator: 1.00 hours
Microwave: 1.00 hours
Oven: 1.00 hours
Lighting: 1.00 hours
TV: 0.25 hours
